# 07 — Deploy Hire Right Databricks App

**Jackson and Jackson HR Digital**

Builds the `app.yaml` configuration, deploys the Hire Right front-end as a Databricks App, grants Unity Catalog permissions to the app service principal, and prints the live URL.

**Prerequisites:** Notebook `06` must have run (agent endpoint deployed).

## Setup

Install the Databricks SDK, configure widgets, and resolve the app source path from the notebook workspace location.

In [ ]:
%pip install -q "databricks-sdk>=0.40.0" pyyaml

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog",             "bx4",                        "UC Catalog")
dbutils.widgets.text("schema",              "hrd_2030",                   "UC Schema")
dbutils.widgets.text("agent_endpoint_name", "hire-right-agent-endpoint",  "Agent Endpoint Name")
dbutils.widgets.text("warehouse_id",        "",                           "SQL Warehouse ID")
dbutils.widgets.text("app_name",            "hire-right-agent-app",       "Databricks App Name")
dbutils.widgets.text("genie_space_id",      "01f13a0f6a081fabbea933cfb0db1d01", "Genie Space ID")

In [ ]:
import os
from databricks.sdk import WorkspaceClient

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = notebook_path.rsplit("/notebooks", 1)[0]  # workspace-relative, no /Workspace prefix
try:
    from dotenv import load_dotenv
    load_dotenv(f"/Workspace{project_root}/.env")
except ImportError:
    pass

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog             = dbutils.widgets.get("catalog")             or os.getenv("TARGET_CATALOG", "bx4")
schema              = dbutils.widgets.get("schema")              or os.getenv("TARGET_SCHEMA", "hrd_2030")
agent_endpoint_name = dbutils.widgets.get("agent_endpoint_name") or os.getenv("DATABRICKS_AGENT_ENDPOINT", "hire-right-agent-endpoint")
warehouse_id        = dbutils.widgets.get("warehouse_id")        or os.getenv("DATABRICKS_WAREHOUSE_ID", "")
app_name            = dbutils.widgets.get("app_name")            or os.getenv("APP_NAME", "hire-right-agent-app")
genie_space_id      = dbutils.widgets.get("genie_space_id")      or os.getenv("GENIE_SPACE_ID", "")

app_path = f"{project_root}/app"
w = WorkspaceClient()

print(f"Catalog             : {catalog}")
print(f"Schema              : {schema}")
print(f"Agent Endpoint      : {agent_endpoint_name}")
print(f"Warehouse ID        : {warehouse_id or '(not set)'}")
print(f"App Name            : {app_name}")
print(f"Project Root        : {project_root}")
print(f"App Source Path     : {app_path}")

## Build App Config

Assemble `app.yaml` with the uvicorn command, environment variables, and resource permissions for the agent endpoint and SQL warehouse.

In [ ]:
import yaml

app_config = {
    "command": ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    "env": [
        {"name": "DATABRICKS_AGENT_ENDPOINT", "value": agent_endpoint_name},
        {"name": "DATABRICKS_WAREHOUSE_ID",   "value": warehouse_id},
        {"name": "GENIE_SPACE_ID",            "valueFrom": "genie_space"},
        {"name": "TARGET_CATALOG",            "value": catalog},
        {"name": "TARGET_SCHEMA",             "value": schema},
    ],
    "resources": [
        {
            "name": "agent_endpoint",
            "serving_endpoint": {
                "name":       agent_endpoint_name,
                "permission": "CAN_QUERY",
            },
        },
        {
            "name": "sql_warehouse",
            "sql_warehouse": {
                "id":         warehouse_id,
                "permission": "CAN_USE",
            },
        },
        {
            "name": "genie_space",
            "genie_space": {
                "name":     "Jackson and Jackson HR Digital \u2014 Hiring Analytics Genie",
                "space_id": genie_space_id,
                "permission": "CAN_RUN",
            },
        },
    ],
}

yaml_content = yaml.dump(app_config, default_flow_style=False)
dest_path    = f"/Workspace{app_path}/app.yaml"

dbutils.fs.put(dest_path, yaml_content, overwrite=True)
print(f"\u2713 app.yaml written to {dest_path}")
print("\n--- app.yaml contents ---")
print(yaml_content)

## Create, Start & Deploy the Databricks App

Create the app if it does not exist, wait for compute to reach RUNNING, then deploy from the workspace source path.

In [ ]:
import time
from databricks.sdk.service.apps import App, AppDeployment, AppDeploymentMode

# ── Step 1: Create app if it doesn't exist ────────────────────────────────────
try:
    w.apps.create(App(name=app_name))
    print(f"\u2713 App created: {app_name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"App '{app_name}' already exists \u2014 will redeploy with updated source.")
    else:
        raise

# ── Step 2: Wait for app compute to reach RUNNING before deploying ─────────────
# A newly created app needs time to provision its compute. Deploying before
# RUNNING raises: 'Cannot deploy app as it is not in RUNNING state'.
print(f"\nWaiting for app compute to be RUNNING (up to 10 min)...")
for attempt in range(60):  # 60 * 10s = 10 min
    info       = w.apps.get(name=app_name)
    app_state  = str(info.app_status.state).upper()  if info.app_status  else "UNKNOWN"
    comp_state = str(info.compute_status.state).upper() if info.compute_status else "UNKNOWN"
    print(f"  [{attempt * 10:>4}s] app={app_state}  compute={comp_state}")
    if "RUNNING" in app_state or "ACTIVE" in comp_state:
        print("\u2713 App is ready")
        break
    time.sleep(10)
else:
    raise TimeoutError(f"App '{app_name}' did not reach RUNNING within 10 minutes.")

# ── Step 3: Deploy ─────────────────────────────────────────────────────────────
print(f"\nDeploying app '{app_name}' from {app_path} ...")
deployment = w.apps.deploy_and_wait(
    app_name=app_name,
    app_deployment=AppDeployment(
        source_code_path=f"/Workspace{app_path}",
        mode=AppDeploymentMode.SNAPSHOT,
    ),
)
print(f"\u2713 App deployed successfully")
print(f"  Deployment ID : {deployment.deployment_id}")
print(f"  Status        : {deployment.status.state if deployment.status else 'N/A'}")

## Grant Unity Catalog Permissions

Grant the app service principal read access to the catalog and schema so it can query the Gold table.

In [ ]:
try:
    import requests as _req
    _ctx2   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    _host   = _ctx2.apiUrl().get().rstrip("/")
    _token  = _ctx2.apiToken().get()
    _resp   = _req.get(f"{_host}/api/2.0/apps/{app_name}",
                       headers={"Authorization": f"Bearer {_token}"}, timeout=30)
    sp_client_id = _resp.json().get("service_principal_client_id") if _resp.ok else None

    if sp_client_id:
        spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{sp_client_id}`")
        spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`")
        spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`")
        print(f"\u2713 UC permissions granted to app service principal: {sp_client_id}")
    else:
        print("App service principal client ID not yet available \u2014 run this cell again after deployment.")
except Exception as e:
    print(f"Permission note (may be safe to ignore if SP not yet provisioned): {e}")

## Confirm App is Running

Poll the app status until it reaches RUNNING state and print the live URL for stakeholders.

In [ ]:
import time

print("Polling app status (checking every 10 s, up to 5 min)...\n")

app_url = None
for attempt in range(30):
    status_info = w.apps.get(name=app_name)
    raw_state   = status_info.app_status.state if status_info.app_status else "UNKNOWN"
    app_status  = str(raw_state).upper()
    print(f"  [{attempt * 10:>4}s] App status: {app_status}")

    if "RUNNING" in app_status:
        app_url = status_info.url
        print(f"\n\u2713 App is RUNNING!")
        print(f"  App URL: {app_url}")
        break

    time.sleep(10)

if not app_url:
    print("\nApp did not reach RUNNING within 5 minutes \u2014 check the Databricks Apps UI.")

## Summary

The Hire Right Databricks App is now live.

| Resource | Value |
|---|---|
| App Name | `hire-right-agent-app` |
| Agent Endpoint | `hire-right-agent-endpoint` |
| Catalog / Schema | `bx4.hrd_2030` |

### Re-deploying after code changes
Run this notebook again — it will overwrite `app.yaml` and trigger a new SNAPSHOT deployment.